# PI3 Grupo 4 — Comparação de discretização: binCount=64 (v2) vs. binWidth=25 HU (v3)

**Objetivo:** isolar o efeito de um único parâmetro — o modo de discretização de
intensidade usado pelo PyRadiomics — mantendo todos os demais parâmetros do
pipeline idênticos aos da Rodada 2 (v2), já validada e documentada no
Relatório de Missão.

### Os dois modos, em resumo

- **binCount = 64** (config atual do grupo): sempre 64 faixas de intensidade,
  não importa a amplitude real (em HU) de cada nódulo. A largura de cada faixa
  varia de nódulo para nódulo.
- **binWidth = 25**: cada faixa sempre representa 25 HU de largura, e o número
  de faixas varia conforme a amplitude de intensidade de cada nódulo.

van Timmeren et al. (2020) recomendam largura fixa para modalidades com escala
física absoluta, como a tomografia computadorizada, e Haarburger et al. (2020)
usaram binWidth=25 no próprio LIDC-IDRI — o valor testado aqui replica essa
escolha, permitindo comparação direta de configuração com aquele trabalho.

### O que muda e o que não muda entre v2 e v3

| Aspecto | v2 (binCount=64) | v3 (binWidth=25) |
|---|---|---|
| Máscara, espaçamento, interpolador | idênticos | idênticos |
| Critérios de elegibilidade do nódulo | idênticos | idênticos |
| Regra de rótulo | idêntica | idêntica |
| Atributos de forma (shape) | não dependem de discretização | devem ser **idênticos** a v2 |
| Atributos de intensidade e textura | calculados com 64 faixas | calculados com faixas de 25 HU |

Se os atributos de forma não baterem exatamente entre v2 e v3, algo além da
discretização mudou entre as rodadas, e isso precisa ser investigado antes de
confiar na comparação.

### Pré-requisito

Este notebook espera que `piloto_piloto_v2_consenso50.csv` já exista na pasta
`features/` do Drive, gerado pela Rodada 2. Ele não recalcula a v2 — apenas a
lê para comparação.

**Nenhum resultado deste notebook foi obtido até você executá-lo.**

## 1. Instalação

Mesmo ambiente da Rodada 2. Um pacote por comando; commit do PyRadiomics
fixado, para que a comparação não seja contaminada por diferença de versão da
biblioteca entre as duas rodadas.

In [ ]:
PYRADIOMICS_COMMIT = '8ed579383'  # o mesmo commit usado na v2

# Mesma ordem da v2: o PyRadiomics compila extensoes C contra o NumPy do
# momento do build, entao as dependencias gerais vem antes e ele por ultimo.
!pip install pylidc
!pip install SimpleITK
!pip install idc-index
!pip install pandas pyarrow scipy

# PyRadiomics por ULTIMO, ja contra o NumPy final do ambiente.
!pip install git+https://github.com/AIM-Harvard/pyradiomics.git@{PYRADIOMICS_COMMIT}

In [ ]:
compat_src = '''
"""compat.py -- restaura APIs removidas, exigidas pelo pylidc 0.2.3."""
import configparser
import numpy as np

if not hasattr(configparser, "SafeConfigParser"):
    configparser.SafeConfigParser = configparser.ConfigParser

for _n, _t in {"float": float, "int": int, "bool": bool,
               "object": object, "str": str, "complex": complex}.items():
    if not hasattr(np, _n):
        setattr(np, _n, _t)

if not hasattr(np, "in1d"):     np.in1d = np.isin
if not hasattr(np, "alltrue"):  np.alltrue = np.all
if not hasattr(np, "sometrue"): np.sometrue = np.any
'''

import sys
with open('/content/compat.py', 'w') as fh:
    fh.write(compat_src)
sys.path.insert(0, '/content')
import compat  # noqa: F401

import numpy as np, configparser


# --- Registro do ambiente desta sessao ---------------------------------
# As versoes sao LIDAS, nunca fixadas. SimpleITK, pandas, pyarrow e pylidc
# nao foram registrados nas rodadas historicas, entao nao existe valor de
# referencia que justifique um pin arbitrario: o que importa e poder saber,
# depois, com o que cada tabela foi produzida. O unico requisito obrigatorio
# continua sendo o PyRadiomics no commit fixado, conferido no fim da celula.
#
# pylidc e consultado pelos METADADOS da distribuicao, nunca por import: o
# modulo le o .pylidcrc no momento em que e importado, e importa-lo aqui,
# antes de o arquivo existir, quebraria a leitura dos volumes DICOM.
from importlib.metadata import PackageNotFoundError, version as _versao_dist

PACOTES_AMBIENTE = ['python', 'numpy', 'pyradiomics', 'simpleitk', 'pandas',
                    'pyarrow', 'pylidc']


def _dist(nome):
    try:
        return _versao_dist(nome)
    except PackageNotFoundError:
        return None


def _runtime_id():
    """Identificador da VM desta sessao.

    Dois notebooks executados no mesmo runtime Colab leem o mesmo boot_id. E
    o que permite comprovar, depois, que a v2 e a v3 do experimento
    definitivo sairam da MESMA sessao, sem reinicio nem reinstalacao.
    """
    try:
        with open('/proc/sys/kernel/random/boot_id') as fh:
            return fh.read().strip()
    except OSError:
        return None


AMBIENTE = {
    'python': sys.version.split()[0],
    'numpy': np.__version__,
    'pyradiomics': _dist('pyradiomics'),
    'simpleitk': _dist('SimpleITK'),
    'pandas': _dist('pandas'),
    'pyarrow': _dist('pyarrow'),
    'pylidc': _dist('pylidc'),
    'runtime_id': _runtime_id(),
}

print('--- ambiente desta sessao ---')
for _k in PACOTES_AMBIENTE:
    print(f'  {_k:<12} {AMBIENTE[_k]}')
print(f"  {'runtime_id':<12} {AMBIENTE['runtime_id']}")

_ausentes = [k for k in PACOTES_AMBIENTE if AMBIENTE[k] is None]
if _ausentes:
    print('  AVISO: versao nao resolvida para', _ausentes)

# Requisito duro: o PyRadiomics tem de ser o build do commit fixado. Um
# release do PyPI produziria features nao comparaveis com as rodadas
# anteriores, e o engano so apareceria la na frente, no controle de Shape.
if PYRADIOMICS_COMMIT not in (AMBIENTE['pyradiomics'] or ''):
    raise RuntimeError(
        f"PyRadiomics resolvido = {AMBIENTE['pyradiomics']!r}, que nao contem "
        f'o commit fixado {PYRADIOMICS_COMMIT!r} (esperado algo como '
        f"'3.1.1.dev111+g{PYRADIOMICS_COMMIT}'). A instalacao a partir do "
        'repositorio falhou e provavelmente caiu num release do PyPI. As '
        'features nao seriam comparaveis com as rodadas anteriores. '
        'Reexecute a celula de instalacao.')

## 2. Drive e caminhos

In [ ]:
import os, glob, hashlib, json, shutil, tarfile, time
import pandas as pd
from collections import Counter
from google.colab import drive


def sha256_arquivo(caminho, bloco=1 << 20):
    """SHA-256 de um arquivo, lido em blocos de 1 MiB.

    Le incrementalmente para nao carregar a tabela inteira na memoria. O hash
    e sempre calculado em runtime, sobre o arquivo realmente usado - nunca
    fixado no codigo.
    """
    h = hashlib.sha256()
    with open(caminho, 'rb') as fh:
        for pedaco in iter(lambda: fh.read(bloco), b''):
            h.update(pedaco)
    return h.hexdigest()

drive.mount('/content/drive')

BASE     = '/content/drive/MyDrive/PI3_Grupo4'
ARQUIVOS = f'{BASE}/dicom_tar'
FEATURES = f'{BASE}/features'
CONFIG   = f'{BASE}/config'
for d in (BASE, ARQUIVOS, FEATURES, CONFIG):
    os.makedirs(d, exist_ok=True)

LIDC_ROOT = '/content/lidc'
os.makedirs(LIDC_ROOT, exist_ok=True)

CSV_V2    = f'{FEATURES}/piloto_piloto_v2_consenso50.csv'
CONFIG_V2 = f'{CONFIG}/config_piloto_v2.json'

for caminho, o_que in ((CSV_V2, 'tabela de features da Rodada 2'),
                       (CONFIG_V2, 'configuracao da Rodada 2')):
    if not os.path.exists(caminho):
        raise FileNotFoundError(
            f'{caminho} nao encontrado ({o_que}). Este notebook compara contra a '
            'Rodada 2; rode o notebook piloto_v2 antes, ou ajuste o caminho acima.')

SHA_CSV_V2    = sha256_arquivo(CSV_V2)
SHA_CONFIG_V2 = sha256_arquivo(CONFIG_V2)

print('Baseline v2 encontrado:')
print(f'  tabela : {CSV_V2}')
print(f'           sha256 {SHA_CSV_V2}')
print(f'  config : {CONFIG_V2}')
print(f'           sha256 {SHA_CONFIG_V2}')

## 3. Parâmetros da Rodada 3 (v3)

Apenas o modo de discretização muda em relação à v2. Todo o resto é copiado
propositalmente igual, para que qualquer diferença encontrada na comparação
seja atribuível unicamente à discretização.

In [ ]:
PARAMS = {
    'versao_config': 'piloto_v3',

    'espacamento': [1.0, 1.0, 1.0],
    'interpolador': 'sitkBSpline',
    'modo_discretizacao': 'binWidth',     # UNICA mudanca em relacao a v2
    'bin_count': 64,                       # ignorado nesta rodada
    'bin_width': 25,                       # usado nesta rodada (HU)

    'mascara': 'consenso50',
    'min_anotadores': 3,
    'diametro_min_mm': 3.0,

    'regra_rotulo': 'mediana',
    'mediana_max_benigno': 2.0,             # protocolo Sec. 5.2: mediana <= 2.0 -> alvo 0
    'mediana_min_maligno': 4.0,             # protocolo Sec. 5.2: mediana >= 4.0 -> alvo 1

    'n_pacientes_piloto': 25,               # mesmos 25 pacientes da v2
    'arquivar_tar_no_drive': True,
    'max_tentativas_extracao': 2,
    'margem_contexto_voxels': 4,            # correcao 13.1 da v2, mantida
}

CLASSES = ['shape', 'firstorder', 'glcm', 'glrlm', 'glszm']

for k, v in PARAMS.items():
    print(f'{k:28s} {v}')

## 4. Obter os dados

Mesma lógica das rodadas anteriores: reaproveita o que já existe local ou no
Drive antes de baixar do zero. Como a v2 já processou estes 25 pacientes, os
`.tar` já devem estar no Drive.

In [ ]:
def inspecionar():
    tars = sorted(glob.glob(f'{ARQUIVOS}/*.tar'))
    dirs = [d for d in os.listdir(LIDC_ROOT)
            if os.path.isdir(os.path.join(LIDC_ROOT, d))] if os.path.exists(LIDC_ROOT) else []
    n_dcm = len(glob.glob(f'{LIDC_ROOT}/**/*.dcm', recursive=True))
    print(f'{len(tars)} tars no Drive | {len(dirs)} pastas locais | {n_dcm} arquivos .dcm')
    return tars, dirs, n_dcm

tars, dirs_existentes, n_dcm = inspecionar()
N = PARAMS['n_pacientes_piloto']

if n_dcm > 0 and len(dirs_existentes) >= N:
    FONTE = 'ja_pronto'
elif tars:
    FONTE = 'tar_drive'
else:
    FONTE = 'idc'
print('fonte:', FONTE)

In [ ]:
if FONTE == 'ja_pronto':
    print('DICOM já local; nada a fazer.')
elif FONTE == 'tar_drive':
    shutil.rmtree(LIDC_ROOT, ignore_errors=True); os.makedirs(LIDC_ROOT, exist_ok=True)
    for t in tars[:N]:
        with tarfile.open(t) as tf:
            tf.extractall(LIDC_ROOT)
    print(f'{min(N, len(tars))} tars descompactados')
elif FONTE == 'idc':
    from idc_index import IDCClient
    client = IDCClient.client()
    inv = client.sql_query("""
        SELECT PatientID, SeriesInstanceUID, Modality, series_size_MB
        FROM index WHERE collection_id = 'lidc_idri' AND Modality = 'CT'
    """)
    alvo = sorted(inv.PatientID.unique())[:N]
    shutil.rmtree(LIDC_ROOT, ignore_errors=True); os.makedirs(LIDC_ROOT, exist_ok=True)
    client.download_from_selection(patientId=alvo, downloadDir=LIDC_ROOT,
        dirTemplate='%PatientID/%StudyInstanceUID/%SeriesInstanceUID')

pids_disco = sorted(d for d in os.listdir(LIDC_ROOT)
                    if os.path.isdir(os.path.join(LIDC_ROOT, d)))
n_dcm = len(glob.glob(f'{LIDC_ROOT}/**/*.dcm', recursive=True))
print(f'{len(pids_disco)} pacientes | {n_dcm} arquivos .dcm')
if n_dcm == 0:
    raise RuntimeError('Nenhum .dcm em disco.')

## 5. Importar pylidc

In [ ]:
RC = os.path.expanduser('~/.pylidcrc')
with open(RC, 'w') as fh:
    fh.write(f'[dicom]\npath = {LIDC_ROOT}\nwarn = True\n')

for m in [m for m in list(sys.modules) if m.split('.')[0] == 'pylidc']:
    del sys.modules[m]

import pylidc as pl
from pylidc.utils import consensus

scans_teste = pl.query(pl.Scan).filter(pl.Scan.patient_id.in_(pids_disco)).all()
v = scans_teste[0].to_volume()
print(f'OK -- {scans_teste[0].patient_id}: volume {v.shape}')

## 6. Configuração do PyRadiomics (v3)

In [ ]:
import logging, radiomics
from radiomics import featureextractor
radiomics.logger.setLevel(logging.ERROR)

CFG = {'resampledPixelSpacing': PARAMS['espacamento'],
       'interpolator': PARAMS['interpolador'],
       'label': 1}
if PARAMS['modo_discretizacao'] == 'binCount':
    CFG['binCount'] = PARAMS['bin_count']
else:
    CFG['binWidth'] = PARAMS['bin_width']

MANIFESTO = {'params_grupo': PARAMS, 'config_pyradiomics': CFG,
             'classes_habilitadas': CLASSES,
             'pyradiomics_commit_fixado': PYRADIOMICS_COMMIT,
             'versao_pyradiomics_resolvida': radiomics.__version__,
             'versao_numpy': np.__version__,
             'versao_python': sys.version.split()[0],
             'ambiente': AMBIENTE,
             'gerado_em': time.strftime('%Y-%m-%d %H:%M:%S')}

with open(f"{CONFIG}/config_{PARAMS['versao_config']}.json", 'w') as fh:
    json.dump(MANIFESTO, fh, indent=2, ensure_ascii=False)

print(json.dumps(CFG, indent=2))

## 7. Extração (mesma lógica robusta da v2)

Inclui as duas correções da Rodada 2: extractor isolado por nódulo e margem de
contexto antes da reamostragem. Sem elas, qualquer diferença encontrada entre
v2 e v3 poderia ser efeito residual de bugs já conhecidos, não da discretização.

In [ ]:
import SimpleITK as sitk

CLEVEL = {'consenso50': 0.5, 'uniao': 0.01, 'intersecao': 1.0}


def montar_mascara(anns, modo):
    if modo in CLEVEL:
        cmask, cbbox, _ = consensus(anns, clevel=CLEVEL[modo])
        return cmask, cbbox
    if modo.startswith('leitor'):
        i = int(modo.replace('leitor', ''))
        if i >= len(anns):
            return None, None
        cmask, cbbox, _ = consensus([anns[i]], clevel=0.5)
        return cmask, cbbox
    raise ValueError(modo)


def rotular(escores, p):
    # retorna (valor_central, alvo, exclusion_reason)
    if p['regra_rotulo'] == 'mediana':
        c = float(np.median(escores))
    elif p['regra_rotulo'] == 'media':
        c = float(np.mean(escores))
    else:
        c = float(Counter(escores).most_common(1)[0][0])
    # Protocolo oficial (docs/protocolo_coorte_target_sprint2.md, Secao 5.2 e 5.3):
    #   mediana <= 2.0            -> alvo 0,    'included'
    #   mediana >= 4.0            -> alvo 1,    'included'
    #   mediana == 3.0            -> alvo None, 'consensus_indeterminate'
    #   mediana 2.5 ou 3.5        -> alvo None, 'fractional_median_even_raters'
    if c <= p['mediana_max_benigno']:
        return c, 0, 'included'
    if c >= p['mediana_min_maligno']:
        return c, 1, 'included'
    if abs(c - 3.0) < 1e-9:
        return c, None, 'consensus_indeterminate'
    return c, None, 'fractional_median_even_raters'


def expandir_bbox(cbbox, vol_shape, margem):
    novo = []
    for eixo, tam in zip(cbbox, vol_shape):
        ini = max(0, eixo.start - margem)
        fim = min(tam, eixo.stop + margem)
        novo.append(slice(ini, fim))
    return tuple(novo)


def extrair_features_isolado(img, msk, cfg, classes):
    ext = featureextractor.RadiomicsFeatureExtractor(**cfg)
    ext.disableAllFeatures()
    for c in classes:
        ext.enableFeatureClassByName(c)
    return ext.execute(img, msk)


def extrair_com_retry(img, msk, cfg, classes, tentativas):
    erro = None
    for _ in range(tentativas):
        try:
            return extrair_features_isolado(img, msk, cfg, classes)
        except Exception as e:
            erro = e
    raise erro


def extrair_scan(scan, p, cfg, classes):
    vol = scan.to_volume()
    esp = [float(scan.pixel_spacing), float(scan.pixel_spacing), float(scan.slice_spacing)]

    linhas, descartes = [], []
    for idx, anns in enumerate(scan.cluster_annotations()):
        nid = f'{scan.patient_id}_N{idx:02d}'
        if len(anns) < p['min_anotadores']:
            descartes.append((nid, 'leitores_insuficientes')); continue
        diams = [float(a.diameter) for a in anns]
        if np.mean(diams) < p['diametro_min_mm']:
            descartes.append((nid, 'diametro_abaixo_do_minimo')); continue
        escores = [int(a.malignancy) for a in anns]
        central, alvo, motivo_alvo = rotular(escores, p)
        # indeterminados NAO sao descartados (protocolo, Secao 8, item 2): ficam
        # na tabela com alvo=None e exclusion_reason preenchido
        try:
            cmask, cbbox = montar_mascara(anns, p['mascara'])
            if cmask is None or cmask.sum() == 0:
                descartes.append((nid, 'mascara_vazia')); continue

            cbbox_exp = expandir_bbox(cbbox, vol.shape, p['margem_contexto_voxels'])
            sub = vol[cbbox_exp]
            mask_exp = np.zeros(sub.shape, dtype=cmask.dtype)
            offset = tuple(a.start - b.start for a, b in zip(cbbox, cbbox_exp))
            slices_orig = tuple(slice(o, o + s) for o, s in zip(offset, cmask.shape))
            mask_exp[slices_orig] = cmask

            valores = sub[mask_exp.astype(bool)]
            if not np.all(np.isfinite(valores)):
                descartes.append((nid, 'intensidade_nao_finita')); continue
            if np.any(np.abs(valores) > 5000):
                descartes.append((nid, 'intensidade_fora_de_faixa_hu')); continue

            img = sitk.GetImageFromArray(np.transpose(sub, (2, 0, 1)).astype(np.float32))
            msk = sitk.GetImageFromArray(np.transpose(mask_exp.astype(np.uint8), (2, 0, 1)))
            img.SetSpacing(esp); msk.SetSpacing(esp)

            feats = extrair_com_retry(img, msk, cfg, classes, p['max_tentativas_extracao'])
        except Exception as e:
            descartes.append((nid, f'erro_extracao:{type(e).__name__}')); continue

        linha = {'nodule_id': nid, 'patient_id': scan.patient_id,
                 'n_anotadores': len(anns),
                 'malignancy_mediana': float(np.median(escores)),
                 'valor_central': central, 'alvo': alvo,
                 'indeterminado': alvo is None,
                 'exclusion_reason': motivo_alvo,
                 'config': p['versao_config']}
        linha.update({k: v2 for k, v2 in feats.items() if not k.startswith('diagnostics')})
        linhas.append(linha)
    return linhas, descartes


print('funções definidas para a extração v3')

## 8. Execução da Rodada 3

In [ ]:
todas_linhas, todos_descartes = [], []
t0 = time.time()

for k, pid in enumerate(pids_disco, 1):
    for scan in pl.query(pl.Scan).filter(pl.Scan.patient_id == pid).all():
        L, D = extrair_scan(scan, PARAMS, CFG, CLASSES)
        todas_linhas.extend(L); todos_descartes.extend(D)
    if k % 5 == 0 or k == len(pids_disco):
        print(f'{k}/{len(pids_disco)} | {len(todas_linhas)} nódulos | '
              f'{(time.time()-t0)/k:.1f}s/paciente')

print(f'\nv3 extraídos: {len(todas_linhas)} | descartados: {len(todos_descartes)}')
if todos_descartes:
    print(dict(Counter(d[1] for d in todos_descartes)))

df_v3 = pd.DataFrame(todas_linhas)
sufixo = f"{PARAMS['versao_config']}_{PARAMS['mascara']}"
CSV_V3    = f'{FEATURES}/piloto_{sufixo}.csv'
CONFIG_V3 = f"{CONFIG}/config_{PARAMS['versao_config']}.json"
df_v3.to_csv(CSV_V3, index=False)
print(f'Salvo: piloto_{sufixo}.csv')

## 9. Comparação v2 (binCount=64) vs. v3 (binWidth=25)

### 9.1 Cobertura: mesmos nódulos nas duas rodadas?

Como os critérios de elegibilidade não dependem de discretização, as duas
rodadas devem extrair exatamente os mesmos nódulos. Qualquer divergência aqui
aponta para um problema não relacionado à discretização em si.

Três camadas de proveniência antes de qualquer cálculo: **9.1a** confere os
parâmetros do baseline, **9.1b** amarra cada CSV ao seu `config_*.json`, e
**9.1c** exige que os dois lados tenham saído da mesma sessão Colab, com as
mesmas versões de pacote.

> **Este é o fluxo definitivo do experimento.** Ele aborta se os ambientes
> divergirem — não basta avisar, porque aí `binCount` × `binWidth` deixaria de
> ser a única variável, que é justamente o defeito diagnosticado na Etapa 2.1.
> Comparar contra uma tabela antiga já versionada, como a `v3_oficial`, segue
> sendo legítimo como **controle externo de regressão** — mas essa checagem é
> feita em análise separada, sobre arquivos já gravados, e **não** por este
> caminho.

In [ ]:
# --- 9.1a Proveniencia do baseline -------------------------------------
# O baseline nao pode ser aceito so por existir um CSV com o nome esperado.
# A Etapa 2.1 mostrou que um CSV v2 gerado ANTES da correcao de margem de
# contexto muda a geometria do recorte e contamina a comparacao: os atributos
# de forma passam a divergir, e o efeito da discretizacao fica confundido.
with open(CONFIG_V2, encoding='utf-8') as fh:
    CFG_V2 = json.load(fh)
PARAMS_V2 = CFG_V2['params_grupo']

TOLERANCIA_SHAPE = 1e-6

# parametros que os dois lados TEM de compartilhar para isolar a discretizacao
GEOMETRIA_COMUM = ['espacamento', 'interpolador', 'mascara', 'min_anotadores',
                   'diametro_min_mm', 'margem_contexto_voxels']
# o que o baseline TEM de ser, por definicao deste experimento
BASELINE_ESPERADO = {'modo_discretizacao': 'binCount', 'bin_count': 64,
                     'margem_contexto_voxels': 4}

problemas = []
for chave, esperado in BASELINE_ESPERADO.items():
    obtido = PARAMS_V2.get(chave, '<AUSENTE>')
    if obtido != esperado:
        problemas.append(f'baseline.{chave} = {obtido!r} (esperado {esperado!r})')
for chave in GEOMETRIA_COMUM:
    a = PARAMS_V2.get(chave, '<AUSENTE>')
    b = PARAMS.get(chave, '<AUSENTE>')
    if a != b:
        problemas.append(f'{chave}: baseline={a!r} vs v3={b!r} (tem de ser igual)')
if PARAMS['modo_discretizacao'] != 'binWidth':
    problemas.append(f"v3.modo_discretizacao = {PARAMS['modo_discretizacao']!r} "
                     "(esperado 'binWidth')")

if problemas:
    raise ValueError(
        'BASELINE v2 INCOMPATIVEL - comparacao abortada.\n  '
        + '\n  '.join(problemas)
        + f'\n\n  tabela: {CSV_V2}\n  config: {CONFIG_V2}\n\n'
        'Para isolar apenas a discretizacao, os dois lados precisam compartilhar '
        'espacamento, interpolador, mascara, criterios de elegibilidade e margem '
        'de contexto, diferindo SOMENTE em modo_discretizacao. Regere a Rodada 2 '
        'com o notebook piloto_v2 estruturalmente corrigido antes de comparar.')

print('Baseline v2 validado contra a configuracao da v3:')
print(f"  geometria comum : espacamento={PARAMS['espacamento']} | "
      f"interpolador={PARAMS['interpolador']} | mascara={PARAMS['mascara']} | "
      f"margem_contexto_voxels={PARAMS['margem_contexto_voxels']}")
print(f"  criterios       : min_anotadores={PARAMS['min_anotadores']} | "
      f"diametro_min_mm={PARAMS['diametro_min_mm']}")
print(f"  discretizacao   : v2 {PARAMS_V2['modo_discretizacao']}"
      f"(bin_count={PARAMS_V2['bin_count']})  x  "
      f"v3 {PARAMS['modo_discretizacao']}(bin_width={PARAMS['bin_width']})")
print('  baseline gerado em:', CFG_V2.get('gerado_em'))

df_v2 = pd.read_csv(CSV_V2)


# --- 9.1b Vinculo CSV <-> config ---------------------------------------
# A checagem 9.1a olha so o JSON de configuracao. Sem esta segunda camada, um
# CSV de outra rodada gravado com o nome esperado passaria despercebido: e
# preciso confirmar que a tabela lida e a que aquele config descreve.
def validar_tabela(df, nome, versao_esperada, caminho):
    erros = []

    if 'nodule_id' not in df.columns:
        erros.append("coluna 'nodule_id' ausente")
    else:
        dups = df['nodule_id'][df['nodule_id'].duplicated()].unique().tolist()
        if dups:
            extra = ' ...' if len(dups) > 5 else ''
            erros.append(f'nodule_id duplicado: {dups[:5]}{extra} '
                         f'({len(dups)} id(s) repetido(s))')

    if 'config' not in df.columns:
        erros.append("coluna 'config' ausente")
    else:
        valores = sorted(str(v) for v in df['config'].dropna().unique())
        if len(valores) != 1:
            erros.append(f'coluna config tem {len(valores)} valores distintos: '
                         f'{valores[:5]} (esperado exatamente 1)')
        elif versao_esperada is not None and valores[0] != str(versao_esperada):
            erros.append(f'config no CSV = {valores[0]!r}, mas o JSON declara '
                         f'versao_config = {versao_esperada!r}')

    if erros:
        raise ValueError(
            f'TABELA {nome} INCOMPATIVEL COM SUA CONFIGURACAO - comparacao '
            'abortada.\n  ' + '\n  '.join(erros)
            + f'\n\n  arquivo: {caminho}\n\n'
            'O CSV e o config_*.json precisam descrever a MESMA rodada. Um CSV '
            'de outra execucao, gravado com o nome esperado, passaria pela '
            'checagem de parametros da Secao 9.1a sem ser detectado.')

    print(f'  {nome}: {len(df)} linhas | nodule_id unico | '
          f"config = {df['config'].iloc[0]!r}")


print('Vinculo CSV <-> config:')
validar_tabela(df_v2, 'baseline v2', PARAMS_V2.get('versao_config'), CSV_V2)
validar_tabela(df_v3, 'v3 desta sessao', PARAMS['versao_config'], CSV_V3)

ids_v2 = set(df_v2.nodule_id)
ids_v3 = set(df_v3.nodule_id)
if len(ids_v2) != len(df_v2) or len(ids_v3) != len(df_v3):
    raise ValueError('nodule_id nao e unico - o merge por nodule_id da Secao '
                     '9.2 produziria linhas duplicadas.')


# --- 9.1c Ambiente das duas extracoes ----------------------------------
# Este notebook E o experimento definitivo de discretizacao, entao aqui se
# aborta, nao se avisa: os dois lados precisam ter saido da MESMA sessao
# Colab, sem reinicio nem reinstalacao entre as extracoes. Ambiente
# diferente significa que binCount x binWidth deixa de ser a unica variavel
# - exatamente o defeito diagnosticado na Etapa 2.1.
#
# A v3_oficial versionada continua valendo como CONTROLE EXTERNO DE
# REGRESSAO, mas esse controle e feito FORA deste fluxo, comparando tabelas
# ja gravadas. Por isso um config sem o campo 'ambiente' (anterior a Etapa
# 2.2B-0) e recusado aqui de proposito: ele nao descreve uma sessao.
AMB_V2 = CFG_V2.get('ambiente')

if AMB_V2 is None:
    ambiente_identico = None
    mesmo_runtime = None
    divergencias_ambiente = None
else:
    divergencias_ambiente = {
        k: {'v2': AMB_V2.get(k), 'v3': AMBIENTE.get(k)}
        for k in PACOTES_AMBIENTE if AMB_V2.get(k) != AMBIENTE.get(k)
    }
    ambiente_identico = not divergencias_ambiente
    mesmo_runtime = (AMB_V2.get('runtime_id') is not None
                     and AMB_V2.get('runtime_id') == AMBIENTE.get('runtime_id'))

runtime_v2 = (AMB_V2 or {}).get('runtime_id')
runtime_v3 = AMBIENTE.get('runtime_id')

impedimentos = []
if AMB_V2 is None:
    impedimentos.append("o config do baseline v2 nao registra 'ambiente' "
                        '(foi gerado antes da Etapa 2.2B-0)')
if runtime_v3 is None:
    impedimentos.append('runtime_id do lado v3 ausente - nao foi possivel ler '
                        '/proc/sys/kernel/random/boot_id nesta maquina')
if AMB_V2 is not None:
    if runtime_v2 is None:
        impedimentos.append('runtime_id do lado v2 ausente no config do baseline')
    if not mesmo_runtime:
        impedimentos.append('os dois lados nao vieram da mesma sessao Colab')
    if not ambiente_identico:
        impedimentos.append('as versoes de pacote diferem entre os dois lados')

print('Ambiente dos dois lados:')
print(f'  runtime v2        : {runtime_v2}')
print(f'  runtime v3        : {runtime_v3}')
print(f'  mesma sessao      : {mesmo_runtime}')
print(f'  versoes identicas : {ambiente_identico}')
for _k, _par in (divergencias_ambiente or {}).items():
    print(f"     {_k}: v2={_par['v2']!r} vs v3={_par['v3']!r}")

if impedimentos:
    if divergencias_ambiente:
        _divs = '\n'.join(
            f"    {_k}: v2={_par['v2']!r} vs v3={_par['v3']!r}"
            for _k, _par in divergencias_ambiente.items())
    elif AMB_V2 is None:
        _divs = '    (o baseline nao registra ambiente - nada a comparar)'
    else:
        _divs = '    (nenhuma - as versoes conferem)'
    raise RuntimeError(
        'AMBIENTE INCOMPATIVEL - experimento definitivo abortado.\n  '
        + '\n  '.join(impedimentos)
        + f'\n\n  runtime v2        : {runtime_v2}'
        + f'\n  runtime v3        : {runtime_v3}'
        + f'\n  mesma sessao      : {mesmo_runtime}'
        + f'\n  versoes identicas : {ambiente_identico}'
        + '\n  divergencias de versao:\n' + _divs
        + f'\n\n  baseline: {CSV_V2}\n\n'
        'A comparacao binCount=64 x binWidth=25 so isola a discretizacao se os '
        'dois lados sairem da MESMA sessao Colab, com as mesmas versoes de '
        'PyRadiomics, NumPy, SimpleITK, pandas, pyarrow e pylidc. Reextraia a '
        'Rodada 2 nesta mesma sessao, antes da Secao 8, e execute de novo.\n\n'
        'Comparar contra uma tabela antiga ja versionada (p.ex. a v3_oficial) e '
        'CONTROLE EXTERNO DE REGRESSAO: legitimo, porem em analise separada, '
        'fora deste fluxo definitivo. Nenhum Spearman sera calculado e nenhum '
        'comparacao_v2_v3_discretizacao.json sera gravado.')

print('OK, os dois lados vieram da mesma sessao Colab e do mesmo ambiente')

print(f'nódulos na v2: {len(ids_v2)} | nódulos na v3: {len(ids_v3)}')
print(f'mesmos IDs: {ids_v2 == ids_v3}')
print(f'só na v2: {ids_v2 - ids_v3}')
print(f'só na v3: {ids_v3 - ids_v2}')

### 9.2 Verificação de controle: atributos de forma devem ser idênticos

Atributos de forma (shape) não dependem da discretização de intensidade — são
calculados diretamente da geometria da máscara. Se eles diferirem entre v2 e
v3, há uma fonte de variação além do parâmetro testado, e a comparação da
Seção 9.3 perde validade até isso ser investigado.

In [ ]:
comuns = sorted(ids_v2 & ids_v3)
v2c = df_v2[df_v2.nodule_id.isin(comuns)].set_index('nodule_id').sort_index()
v3c = df_v3[df_v3.nodule_id.isin(comuns)].set_index('nodule_id').sort_index()

cols_shape = [c for c in v2c.columns if '_shape_' in c and c in v3c.columns]
diff_shape = (v2c[cols_shape] - v3c[cols_shape]).abs()
max_diff_shape = diff_shape.max().max()

print(f'{len(cols_shape)} atributos de forma comparados em {len(comuns)} nódulos')
print(f'maior diferença absoluta em atributos de forma: {max_diff_shape:.2e}')

# Fail-fast: atributos de forma sao calculados a partir da geometria da mascara
# e NAO dependem da discretizacao de intensidade. Se divergem, ha uma segunda
# variavel mudando entre os dois lados e a comparacao binCount x binWidth fica
# confundida com esse efeito. Nesse caso NADA e exportado.
if max_diff_shape >= TOLERANCIA_SHAPE:
    pior_attr = diff_shape.max().idxmax()
    pior_nid = diff_shape[pior_attr].idxmax()
    n_divergentes = int((diff_shape >= TOLERANCIA_SHAPE).any().sum())
    raise RuntimeError(
        'CONTROLE DE SHAPE REPROVADO - comparacao de discretizacao INVALIDA.\n'
        f'  maior |diferenca|: {max_diff_shape:.6g} '
        f'(tolerancia {TOLERANCIA_SHAPE:g})\n'
        f'  pior caso: {pior_attr} em {pior_nid}\n'
        f'  atributos de forma divergentes: {n_divergentes} de {len(cols_shape)}\n\n'
        'Atributos de forma dependem apenas da geometria da mascara, nao da '
        'discretizacao de intensidade. Divergencia aqui significa que os dois '
        'lados diferem em algo alem do parametro testado - tipicamente a margem '
        'de contexto ou o recorte/reamostragem. Nenhum arquivo '
        'comparacao_v2_v3_discretizacao.json sera gravado. Corrija a origem do '
        'baseline e repita a comparacao.')

print('OK, forma invariante entre os dois lados - a comparacao isola a discretizacao')

### 9.3 Efeito da discretização sobre intensidade e textura

Para as classes que dependem de discretização (first order, GLCM, GLRLM,
GLSZM), calcula-se a diferença percentual média entre v2 e v3 e a correlação
de Spearman entre os dois conjuntos de valores, por classe de atributo. Uma
correlação alta com diferença percentual moderada indica que a ordenação dos
nódulos pelo atributo se preserva, ainda que a escala numérica mude; uma
correlação baixa indica que a escolha de discretização afeta não apenas a
escala, mas a própria relação entre os nódulos.

In [ ]:
from scipy.stats import spearmanr

classes_dependentes = ['firstorder', 'glcm', 'glrlm', 'glszm']
resumo_classes = []

for classe in classes_dependentes:
    cols = [c for c in v2c.columns if f'_{classe}_' in c and c in v3c.columns]
    if not cols:
        continue
    corrs, difs_pct = [], []
    for c in cols:
        a, b = v2c[c].values, v3c[c].values
        if np.all(a == a[0]) or np.all(b == b[0]):
            continue  # evita erro de correlacao em coluna constante
        rho, _ = spearmanr(a, b)
        corrs.append(rho)
        denom = np.where(np.abs(a) > 1e-9, np.abs(a), np.nan)
        difs_pct.append(np.nanmean(np.abs(a - b) / denom) * 100)
    resumo_classes.append({
        'classe': classe,
        'atributos_comparados': len(cols),
        'spearman_medio': round(float(np.nanmean(corrs)), 3) if corrs else None,
        'spearman_minimo': round(float(np.nanmin(corrs)), 3) if corrs else None,
        'diferenca_pct_media': round(float(np.nanmean(difs_pct)), 1) if difs_pct else None,
    })

df_resumo_classes = pd.DataFrame(resumo_classes)
print(df_resumo_classes.to_string(index=False))

### 9.4 Atributos individuais mais e menos afetados

Identifica, entre todos os atributos dependentes de discretização, quais
mudaram mais e quais mudaram menos ao trocar binCount por binWidth — útil para
decidir se algum atributo específico deveria ser tratado com cautela adicional
na etapa de seleção de atributos.

In [ ]:
cols_dep = [c for c in v2c.columns
            if any(f'_{cl}_' in c for cl in classes_dependentes) and c in v3c.columns]

linhas = []
for c in cols_dep:
    a, b = v2c[c].values, v3c[c].values
    if np.all(a == a[0]) or np.all(b == b[0]):
        continue
    rho, _ = spearmanr(a, b)
    denom = np.where(np.abs(a) > 1e-9, np.abs(a), np.nan)
    dif_pct = np.nanmean(np.abs(a - b) / denom) * 100
    linhas.append({'atributo': c, 'spearman': round(rho, 3), 'diferenca_pct_media': round(dif_pct, 1)})

df_atributos = pd.DataFrame(linhas).sort_values('spearman')

print('=== 10 atributos com MENOR correlação entre v2 e v3 (mais sensíveis à discretização) ===')
print(df_atributos.head(10).to_string(index=False))
print('\n=== 10 atributos com MAIOR correlação entre v2 e v3 (mais estáveis à discretização) ===')
print(df_atributos.tail(10).to_string(index=False))

## 10. Exportação para o relatório

In [ ]:
# Os SHA-256 abaixo sao calculados AGORA, em runtime, sobre os arquivos
# efetivamente usados - nunca fixados no codigo. Se o baseline mudou no disco
# durante a sessao, o JSON descreveria dados que nao foram os comparados.
if (sha256_arquivo(CSV_V2) != SHA_CSV_V2
        or sha256_arquivo(CONFIG_V2) != SHA_CONFIG_V2):
    raise RuntimeError(
        'O baseline v2 mudou no disco durante esta sessao (SHA-256 diferente '
        'do calculado na Secao 3). A comparacao nao descreveria os dados '
        'efetivamente usados. Reexecute o notebook desde o inicio.')

resultado = {
    'comparacao': 'binCount=64 (v2) vs binWidth=25 HU (v3)',
    'pacientes': N,
    'nodulos_comuns': len(comuns),
    'mesmos_ids': ids_v2 == ids_v3,
    'shape_max_diff_absoluta': float(max_diff_shape),
    'shape_invariante': bool(max_diff_shape < TOLERANCIA_SHAPE),
    'tolerancia_shape': TOLERANCIA_SHAPE,
    'proveniencia': {
        'baseline_v2': {
            'tabela': CSV_V2,
            'tabela_sha256': sha256_arquivo(CSV_V2),
            'tabela_linhas': int(len(df_v2)),
            'config': CONFIG_V2,
            'config_sha256': sha256_arquivo(CONFIG_V2),
            'versao_config_no_csv': str(df_v2['config'].iloc[0]),
            'versao_config_no_json': PARAMS_V2.get('versao_config'),
            'gerado_em': CFG_V2.get('gerado_em'),
            'pyradiomics': CFG_V2.get('versao_pyradiomics_resolvida'),
            'ambiente': AMB_V2,
        },
        'v3': {
            'tabela': CSV_V3,
            'tabela_sha256': sha256_arquivo(CSV_V3),
            'tabela_linhas': int(len(df_v3)),
            'config': CONFIG_V3,
            'config_sha256': sha256_arquivo(CONFIG_V3),
            'versao_config_no_csv': str(df_v3['config'].iloc[0]),
            'versao_config_no_json': PARAMS['versao_config'],
            'origem': 'extraido nesta sessao (Secao 8 deste notebook)',
            'gerado_em': MANIFESTO.get('gerado_em'),
            'pyradiomics': MANIFESTO.get('versao_pyradiomics_resolvida'),
            'ambiente': AMBIENTE,
        },
        'mesma_sessao_colab': mesmo_runtime,
        'ambiente_identico': ambiente_identico,
        'divergencias_de_ambiente': divergencias_ambiente,
        'parametros_geometricos_comuns': {k: PARAMS[k] for k in GEOMETRIA_COMUM},
        'parametros_de_discretizacao_diferentes': {
            'v2': {'modo_discretizacao': PARAMS_V2['modo_discretizacao'],
                   'bin_count': PARAMS_V2['bin_count']},
            'v3': {'modo_discretizacao': PARAMS['modo_discretizacao'],
                   'bin_width': PARAMS['bin_width']},
        },
    },
    'resumo_por_classe': df_resumo_classes.to_dict(orient='records'),
    'atributos_menos_estaveis': df_atributos.head(10).to_dict(orient='records'),
    'atributos_mais_estaveis': df_atributos.tail(10).to_dict(orient='records'),
}

with open(f'{FEATURES}/comparacao_v2_v3_discretizacao.json', 'w') as fh:
    json.dump(resultado, fh, indent=2, ensure_ascii=False)

print(json.dumps({k: v for k, v in resultado.items()
                  if k not in ('atributos_menos_estaveis', 'atributos_mais_estaveis')},
                 indent=2, ensure_ascii=False))
print('\nSalvo em features/comparacao_v2_v3_discretizacao.json')

---
### Como ler o resultado

**Se a Seção 9.1 mostrar `mesmos_ids: False`:** algo além da discretização
mudou entre as rodadas (possivelmente um problema de ambiente); investigar
antes de interpretar qualquer coisa das seções seguintes.

**Se a Seção 9.2 mostrar `shape_invariante: False`:** o mesmo alerta se aplica
— atributos de forma não deveriam mudar com discretização, e uma divergência
ali é sinal de erro, não de efeito real do parâmetro testado.

**Se ambas as verificações de controle passarem**, a Seção 9.3 é a análise que
interessa: uma correlação de Spearman alta (próxima de 1) entre v2 e v3 indica
que a ordenação relativa dos nódulos por aquele atributo é preservada mesmo
quando a escala muda — o que sustentaria manter esse atributo na seleção,
independentemente de qual modo de discretização o grupo adotar na versão
final. Correlação baixa em algum atributo específico merece nota à parte na
discussão de seleção de atributos, e não deveria ser combinado com resultados
de outra configuração sem essa ressalva.